# Stage 3: custom PyTorch model verification

Training-only correctness, tiny-subset learning and CPU runtime diagnostics. No baseline training, tuning or test scoring. Stage 2 remains read-only. The runner preserves previous Stage 3 outputs before reruns. Reusable models and masking are in `src/models.py`; diagnostic procedures are in `src/stage3.py`.

Both architectures use random initialisation, a 13,898 x 64 embedding and a three-logit head. CNN: width-3 convolution with 64 filters, ReLU, masked max pooling, dropout 0.3. GRU: one 64-unit recurrent layer with packed lengths, final hidden state and dropout 0.3. PAD is fixed zero. Word masking is 2% on eligible training words only, reproducible by seed/epoch/source row/position, protecting negation, punctuation, emojis and reserved symbols.

The primary marking scheme assigns 20/20/30/10/10/10 marks; see `docs/stage3_proposal.md` for brief sections and Lecture 6/13 transcript page references and caveats. This notebook covers implementation diagnostics, not the full building/tuning/evaluation requirements.

In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd().resolve()
if not (ROOT/'src/models.py').exists(): ROOT=ROOT.parent
assert Path(sys.prefix).resolve()==(ROOT/'.venv').resolve()
sys.path.insert(0,str(ROOT/'src'))
from stage3 import run
print('Project Python:',sys.executable)
report=run()
assert report['all_passed']

Project Python: <local-path-redacted>

cnn verified; diagnostic accuracy 1.0 mean batch seconds 0.013274465000358759


gru verified; diagnostic accuracy 1.0 mean batch seconds 0.02640062999962538


## Actual verification results

The parameter counts below are measured from instantiated models and include 64 fixed PAD entries. UNK gradients/updates come from actual 2% masking on training examples. The 24-example balanced overfit check uses a separate fresh model, plain cross-entropy, learning rate 0.01 and disables dropout/masking to isolate learning ability. Its accuracy is not a generalisation estimate. All diagnostic weights are discarded; no checkpoint is saved.

In [2]:
import pandas as pd
from IPython.display import display
display(pd.Series(report['checks'],name='passed').to_frame())
display(pd.DataFrame({name:{'parameters':r['parameters'],'padding_difference':r['padding_max_abs_difference'],
    'UNK_gradient_norm':r['unk_gradient_norm'],'UNK_update_norm':r['unk_update_norm'],
    'tiny_subset_accuracy':r['overfit_diagnostic']['training_subset_accuracy'],
    'tiny_initial_loss':r['overfit_diagnostic']['initial_loss'],'tiny_final_loss':r['overfit_diagnostic']['final_loss'],
    'tiny_steps':r['overfit_diagnostic']['steps']} for name,r in report['models'].items()}).T)
display(report['masking'])

,passed
mask_input_unchanged,True
protected_tokens_unchanged,True
mask_exposure_positive,True
mask_reproducible,True
mask_batch_order_independent,True
masking_disabled_validation_True,True
masking_disabled_test_True,True
masking_disabled_train_False,True
explicit_protected_negation_punctuation_emoji,True
cnn_parameter_count,True


,parameters,padding_difference,UNK_gradient_norm,UNK_update_norm,tiny_subset_accuracy,tiny_initial_loss,tiny_final_loss,tiny_steps
cnn,902019.0,0.0,0.017476,0.008000,1.0,1.116877,0.039645,5.0
gru,914627.0,0.0,0.004215,0.007999,1.0,1.102492,0.001326,10.0


{'probability': 0.02,
 'eligible_training_positions': 162393,
 'masked_positions_epoch0': 3261,
 'observed_fraction': 0.020080914817756923,
 'protected_policy': 'reserved IDs, negation/nt contractions, punctuation, emojis',
 'seed': 42,
 'scope': 'training only; same stateless policy for both models'}

## CPU timing

Fresh models, one at a time: five warm-up then twenty timed training batches, batch size 32, two CPU threads, float32 and no loader workers. Timing includes batch construction, masking, forward/backward, gradient clipping and Adam updates. It excludes initialisation, validation and checkpoints. Epoch estimates use 313 batches and are approximate; laptop load and batch lengths vary.

In [3]:
display(report['environment'])
display(pd.DataFrame({name:{k:v for k,v in r['cpu_timing'].items() if k!='batch_seconds'} for name,r in report['models'].items()}).T)
assert report['diagnostic_weights_discarded'] and not report['weights_saved']
assert not report['baseline_training_started'] and not report['test_performance_accessed']
print('Stage 2 hashes preserved. Diagnostic weights discarded. Baseline training pending review.')

{'python': '3.13.7',
 'torch': '2.14.0+cpu',
 'platform': 'Windows-11-10.0.26200-SP0',
 'processor': 'Intel64 Family 6 Model 186 Stepping 2, GenuineIntel',
 'device': 'cpu',
 'threads': 2}

,warmup_batches,timed_batches,batch_size,threads,loader_workers,dtype,mean_batch_seconds,median_batch_seconds,minimum_batch_seconds,maximum_batch_seconds,estimated_313_batch_training_epoch_seconds,estimate_excludes,masked_positions_25_batches
cnn,5,20,32,2,0,float32,0.013274,0.011649,0.008284,0.039074,4.154908,"validation, checkpoints, startup; short sample...",241
gru,5,20,32,2,0,float32,0.026401,0.026735,0.020214,0.034397,8.263397,"validation, checkpoints, startup; short sample...",241


Stage 2 hashes preserved. Diagnostic weights discarded. Baseline training pending review.


## Pending baseline plan and assistance record

See `STAGE3.md` for the proposed shared loss/optimiser/batch/early-stopping settings. Future training must call `fresh_model` and create a new optimiser, reset seeds and use only validation macro-F1 for early stopping. Test scoring remains deferred.

Codex generated and executed model, masking and diagnostic code and prepared working result notes. This includes diagnostic training on training-only subsets. No baseline training, tuning, held-out performance evaluation, pretrained weights or assignment report was produced. Any later disclosure must accurately describe this assistance under applicable requirements.